# 02 — Data Quality, EDA & Organization-Year Deduplication

IRS Form 990 significant diversion disclosures, tax years 2023–2025. Reuses Notebook 01 Parquet partitions; does not download or parse XML. **Labels are self-reported disclosures, not confirmed fraud.**

## 1 — Setup and locate ingestion outputs

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / ".gitignore").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open JupyterLab from within the irs-990-fraud-detection repository")

STAGING = ROOT / "data" / "staging" / "form990_xml_partitions"
REPORTS = ROOT / "reports" / "ingestion"
CLEANED = ROOT / "data" / "cleaned"
ANALYTICS = ROOT / "data" / "analytics"
for folder in (CLEANED, ANALYTICS):
    folder.mkdir(parents=True, exist_ok=True)

TARGET_YEARS = {2023, 2024, 2025}
parquet_paths = sorted(STAGING.glob("*_full.parquet"))
report_paths = sorted(REPORTS.glob("*_full.json"))
print(f"Project: {ROOT}")
print(f"Full Parquet partitions: {len(parquet_paths)} | Full archive reports: {len(report_paths)}")
if not parquet_paths or not report_paths:
    raise FileNotFoundError("Full ingestion outputs not found. Check Notebook 01 output directories.")

## 2 — Validate archive completeness and load filings

In [ ]:
reports = pd.DataFrame([json.loads(p.read_text()) for p in report_paths])
if reports["archive_name"].duplicated().any():
    raise ValueError("Duplicate archive names in ingestion reports")

expected_partitions = {Path(p).stem for p in parquet_paths}
reported_partitions = {Path(p).stem for p in reports["parquet_path"]}
if expected_partitions != reported_partitions:
    raise ValueError(
        f"Parquet/report mismatch: missing={sorted(reported_partitions - expected_partitions)}, "
        f"unreported={sorted(expected_partitions - reported_partitions)}"
    )
if not reports["is_full_archive"].all():
    raise ValueError("A report is marked as a partial ingestion")

filings = pd.concat((pd.read_parquet(p) for p in parquet_paths), ignore_index=True)
for col in ("ein", "filing_id", "source_xml", "organization_name"):
    filings[col] = filings[col].astype("string").str.strip()
filings["ein"] = filings["ein"].str.zfill(9)
filings["tax_year"] = pd.to_numeric(filings["tax_year"], errors="coerce").astype("Int64")
filings["fraud_label"] = pd.to_numeric(filings["fraud_label"], errors="coerce").astype("Int8")

expected_rows = sum(int(r.get("retained", 0)) for r in reports["counts"])
expected_yes = int(reports["positive"].sum())
expected_no = int(reports["negative"].sum())
expected_missing = int(reports["missing_label"].sum())
parse_errors = sum(int(r.get("parse_errors", 0)) for r in reports["counts"])
actual = {
    "retained": len(filings),
    "positive": int(filings["fraud_label"].eq(1).sum()),
    "negative": int(filings["fraud_label"].eq(0).sum()),
    "missing": int(filings["fraud_label"].isna().sum()),
}
expected = {"retained": expected_rows, "positive": expected_yes,
            "negative": expected_no, "missing": expected_missing}
comparison = pd.DataFrame({"from_reports": expected, "from_parquet": actual})
comparison["match"] = comparison["from_reports"].eq(comparison["from_parquet"])
display(comparison)
print(f"Archive reports: {len(reports)} | XML parse errors: {parse_errors:,}")
if not comparison["match"].all():
    raise ValueError("Ingestion totals do not reconcile; investigate before continuing")
if not filings["tax_year"].isin(TARGET_YEARS).all():
    raise ValueError("Out-of-scope or missing tax years found")
if not filings["fraud_label"].isin([0, 1]).all():
    raise ValueError("Missing or invalid disclosure labels found")

## 3 — Data quality and missing values

In [ ]:
quality = pd.DataFrame({
    "missing_count": filings.isna().sum(),
    "missing_pct": (filings.isna().mean() * 100).round(2),
    "unique_non_null": filings.nunique(dropna=True),
}).sort_values("missing_pct", ascending=False)
display(quality)

invalid_ein = filings["ein"].notna() & ~filings["ein"].str.fullmatch(r"\d{9}").fillna(False)
print(f"Invalid EIN format: {int(invalid_ein.sum()):,}")
print(f"Missing filing IDs: {int(filings['filing_id'].isna().sum()):,}")
print(f"Missing organization names: {int(filings['organization_name'].isna().sum()):,}")
print(f"Disclosure field absent: {int((filings['disclosure_field_found'] == False).sum()):,}")

## 4 — Tax-year coverage and disclosure rates

In [ ]:
year_summary = (filings.groupby("tax_year", dropna=False)
    .agg(filings=("fraud_label", "size"),
         positive=("fraud_label", lambda s: int(s.eq(1).sum())),
         negative=("fraud_label", lambda s: int(s.eq(0).sum())))
    .reset_index())
year_summary["positive_rate_pct"] = (100 * year_summary["positive"] / year_summary["filings"]).round(4)
display(year_summary)
print(f"Overall disclosure rate: {100 * actual['positive'] / len(filings):.4f}%")

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].bar(year_summary["tax_year"].astype(str), year_summary["filings"])
axes[0].set(title="Filings by tax year", xlabel="Tax year", ylabel="Filings")
axes[1].bar(year_summary["tax_year"].astype(str), year_summary["positive_rate_pct"])
axes[1].set(title="Positive disclosure rate", xlabel="Tax year", ylabel="Percent")
plt.tight_layout()
plt.show()

## 5 — Duplicates and repeated filings

A repeated **filing ID** may be the same XML appearing in multiple archives. A repeated **EIN + tax year** may be an amended filing or another legitimate repeat. We remove only exact repeated filing IDs from the filing-level output; **we do not automatically collapse EIN + tax-year groups**.

In [ ]:
# Review repeated filing IDs before selecting one copy.
valid_ids = filings["filing_id"].notna()
repeated_ids = filings.loc[valid_ids & filings.duplicated("filing_id", keep=False)].copy()
conflicting_ids = (repeated_ids.groupby("filing_id")
    .agg(ein_values=("ein", "nunique"), tax_year_values=("tax_year", "nunique"),
         label_values=("fraud_label", "nunique")))
conflicting_ids = conflicting_ids[
    (conflicting_ids["ein_values"] > 1) |
    (conflicting_ids["tax_year_values"] > 1) |
    (conflicting_ids["label_values"] > 1)
]
print(f"Rows with repeated filing IDs: {len(repeated_ids):,}")
print(f"Repeated filing IDs with conflicting EIN/year/label: {len(conflicting_ids):,}")
if len(conflicting_ids):
    display(conflicting_ids.head(20))
    raise ValueError("Conflicting filing IDs found; review before deduplication")

# Preserve missing filing IDs as separate records; select a deterministic copy for repeated IDs.
filing_level = filings.sort_values(["publication_year", "archive_name", "source_xml"], kind="stable")
filing_level = pd.concat([
    filing_level[filing_level["filing_id"].notna()].drop_duplicates("filing_id", keep="first"),
    filing_level[filing_level["filing_id"].isna()],
], ignore_index=True)

has_ein_year = filing_level["ein"].notna() & filing_level["tax_year"].notna()
repeated_ein_year = filing_level.loc[
    has_ein_year & filing_level.duplicated(["ein", "tax_year"], keep=False)
].sort_values(["ein", "tax_year", "filing_id"])

print(f"Unique filing-level rows: {len(filing_level):,}")
print(f"Rows in repeated EIN + tax-year groups: {len(repeated_ein_year):,}")
print(f"Repeated EIN + tax-year groups: {repeated_ein_year.groupby(['ein', 'tax_year']).ngroups:,}")
display(repeated_ein_year[["ein", "tax_year", "filing_id", "organization_name",
                          "fraud_label", "archive_name"]].head(15))

## 6 — Check whether duplicate groups disagree on the disclosure label

In [ ]:
if repeated_ein_year.empty:
    label_conflicts = pd.DataFrame(columns=["ein", "tax_year", "filing_count", "distinct_labels", "positive_filings"])
else:
    label_conflicts = (repeated_ein_year.groupby(["ein", "tax_year"])
        .agg(filing_count=("filing_id", "size"),
             distinct_labels=("fraud_label", "nunique"),
             positive_filings=("fraud_label", lambda s: int(s.eq(1).sum())))
        .query("distinct_labels > 1")
        .reset_index())
print(f"EIN + tax-year groups with conflicting disclosure labels: {len(label_conflicts):,}")
display(label_conflicts.head(20))

filing_year_summary = (filing_level.groupby("tax_year")
    .agg(filings=("fraud_label", "size"),
         positive=("fraud_label", lambda s: int(s.eq(1).sum())))
    .reset_index())
filing_year_summary["positive_rate_pct"] = (
    100 * filing_year_summary["positive"] / filing_year_summary["filings"]
).round(4)
display(filing_year_summary)

## 7 — Publication-year coverage

Publication year is when the IRS released the ZIP, **not** the return’s tax year. In particular, later publication archives may contain earlier tax-year filings.

In [ ]:
coverage = pd.crosstab(filing_level["publication_year"], filing_level["tax_year"])
coverage.index.name = "Publication year"
coverage.columns.name = "Tax year"
display(coverage)

## 8 — Inspect repeated organization-year filings

Review all conflicting disclosures and compare tax-period dates, publication year, and filing IDs. The existing dataset does **not** contain a verified amendment flag or filing acceptance timestamp. Publication year and filing ID are only deterministic selection proxies; neither proves which return supersedes another.

In [ ]:
KEYS = ["ein", "tax_year"]
valid_key = filing_level["ein"].str.fullmatch(r"\d{9}").fillna(False) & filing_level["tax_year"].notna()
valid = filing_level.loc[valid_key].copy()
invalid_key_rows = filing_level.loc[~valid_key].copy()

# Preserve the entire filing-level history for audit, including all conflicting cases.
conflict_keys = label_conflicts[KEYS].copy()
conflict_details = valid.merge(conflict_keys, on=KEYS, how="inner")
review_cols = ["ein", "tax_year", "organization_name", "filing_id",
               "publication_year", "archive_name", "tax_period_begin",
               "tax_period_end", "fraud_label", "raw_diversion_response"]
conflict_details = conflict_details.sort_values(
    ["ein", "tax_year", "publication_year", "filing_id"], kind="stable"
)
print(f"Valid-key filing rows: {len(valid):,}")
print(f"Rows with missing/invalid organization-year keys (quarantined): {len(invalid_key_rows):,}")
print(f"Conflicting organization-year groups: {len(conflict_keys):,}")
display(conflict_details[review_cols].head(40))


## 9 — Select a provisional canonical filing per EIN + tax year

**Selection rule:** latest available tax-period end, then latest publication year, then highest filing ID, with source fields as stable tie-breakers. Missing dates sort first. **No disclosure label is used to choose the filing.** This is a reproducible provisional choice, not verification that a filing is amended or authoritative. Preserve `ever_positive` separately and mark every conflicting group for manual review.

In [ ]:
# Group-level metadata are calculated independently of which filing is selected.
group_stats = (valid.groupby(KEYS, dropna=False)
    .agg(filing_count=("fraud_label", "size"),
         distinct_labels=("fraud_label", "nunique"),
         positive_filing_count=("fraud_label", lambda x: int(x.eq(1).sum())))
    .reset_index())
group_stats["has_multiple_filings"] = group_stats["filing_count"].gt(1)
group_stats["has_conflicting_labels"] = group_stats["distinct_labels"].gt(1)
group_stats["ever_positive"] = group_stats["positive_filing_count"].gt(0)

valid["_period_end_sort"] = pd.to_datetime(valid["tax_period_end"], errors="coerce")
valid["_publication_sort"] = pd.to_numeric(valid["publication_year"], errors="coerce")
valid["_filing_sort"] = valid["filing_id"].fillna("")
ordered = valid.sort_values(
    ["_period_end_sort", "_publication_sort", "_filing_sort", "archive_name", "source_xml"],
    ascending=[False, False, False, False, False], na_position="last", kind="stable"
)
canonical = ordered.drop_duplicates(KEYS, keep="first").drop(
    columns=["_period_end_sort", "_publication_sort", "_filing_sort"]
).merge(group_stats, on=KEYS, how="left", validate="one_to_one")
canonical["selection_method"] = "latest_period_end_then_publication_year_then_filing_id"
canonical["selection_verified_amendment"] = False

assert not canonical.duplicated(KEYS).any()
assert len(canonical) == len(group_stats)
assert int(canonical["filing_count"].sum()) == len(valid)

print(f"Canonical organization-year records: {len(canonical):,}")
print(f"Selected-filing positive disclosures: {int(canonical.fraud_label.eq(1).sum()):,}")
print(f"Organization-years with any positive filing: {int(canonical.ever_positive.sum()):,}")
print(f"Organization-years with conflicting labels: {int(canonical.has_conflicting_labels.sum()):,}")
display(canonical.groupby("tax_year").agg(
    organization_years=("ein", "size"),
    selected_positive=("fraud_label", lambda x: int(x.eq(1).sum())),
    ever_positive=("ever_positive", "sum"),
    conflicts=("has_conflicting_labels", "sum")
).reset_index())


## 10 — Create a conservative modeling candidate dataset

Exclude conflicting-label organization-years and invalid keys from the *candidate* modeling dataset until manually resolved. The canonical table still contains them with flags. This dataset is **not yet feature-complete or ready for model training**; financial features and leakage-safe splits come later.

In [ ]:
model_candidates = canonical.loc[~canonical["has_conflicting_labels"]].copy()
assert not model_candidates.duplicated(KEYS).any()
assert not model_candidates["has_conflicting_labels"].any()
assert len(model_candidates) + int(canonical["has_conflicting_labels"].sum()) == len(canonical)
print(f"Modeling candidate organization-years: {len(model_candidates):,}")
print(f"Positive candidate labels: {int(model_candidates.fraud_label.eq(1).sum()):,}")
print(f"Excluded ambiguous organization-years: {int(canonical.has_conflicting_labels.sum()):,}")
print("Note: split by EIN (not randomly by filing) to reduce entity leakage.")


## 11 — Save audited datasets and methodology

Existing filing-level output is retained. New outputs are written separately so no raw data or earlier audit is overwritten. Conflicting filing details and invalid-key records remain available for review.

In [ ]:
filing_output = CLEANED / "form990_filing_level_2023_2025.parquet"
canonical_output = CLEANED / "form990_organization_year_provisional_2023_2025.parquet"
model_output = CLEANED / "form990_model_candidates_2023_2025.parquet"
duplicate_output = ANALYTICS / "form990_repeated_ein_tax_year.parquet"
conflicts_output = ANALYTICS / "form990_label_conflicts.parquet"
conflict_details_output = ANALYTICS / "form990_conflicting_filing_details.csv"
invalid_output = ANALYTICS / "form990_invalid_organization_year_keys.parquet"
summary_output = ANALYTICS / "02_data_quality_summary.json"

filing_level.to_parquet(filing_output, index=False)
canonical.to_parquet(canonical_output, index=False)
model_candidates.to_parquet(model_output, index=False)
repeated_ein_year.to_parquet(duplicate_output, index=False)
label_conflicts.to_parquet(conflicts_output, index=False)
conflict_details[review_cols].to_csv(conflict_details_output, index=False)
invalid_key_rows.to_parquet(invalid_output, index=False)

summary = {
    "archive_reports": len(reports), "parse_errors": parse_errors,
    "raw_filing_rows": len(filings), "raw_positive": actual["positive"],
    "raw_negative": actual["negative"], "raw_missing_labels": actual["missing"],
    "filing_level_rows": len(filing_level),
    "filing_level_positive": int(filing_level.fraud_label.eq(1).sum()),
    "repeated_filing_id_rows": len(repeated_ids),
    "repeated_ein_tax_year_rows": len(repeated_ein_year),
    "repeated_ein_tax_year_groups": int(repeated_ein_year.groupby(KEYS).ngroups),
    "conflicting_ein_tax_year_groups": len(label_conflicts),
    "invalid_key_filing_rows": len(invalid_key_rows),
    "canonical_organization_years": len(canonical),
    "canonical_selected_positive": int(canonical.fraud_label.eq(1).sum()),
    "organization_years_ever_positive": int(canonical.ever_positive.sum()),
    "model_candidate_organization_years": len(model_candidates),
    "model_candidate_positive": int(model_candidates.fraud_label.eq(1).sum()),
    "canonical_rule": "latest tax_period_end; latest publication_year; highest filing_id; stable source tie-breakers",
    "canonical_rule_limitations": "Provisional: amendment status and authoritative filing sequence not verified",
    "conflict_policy": "Retain in canonical with flags; exclude from model candidates pending review",
    "label_definition": "Self-reported significant diversion of assets, not verified fraud",
}
summary_output.write_text(json.dumps(summary, indent=2))
for path in [filing_output, canonical_output, model_output, duplicate_output,
             conflicts_output, conflict_details_output, invalid_output, summary_output]:
    print("Saved:", path.relative_to(ROOT))

# Confirm output row counts after serialization.
assert len(pd.read_parquet(canonical_output, columns=["ein"])) == len(canonical)
assert len(pd.read_parquet(model_output, columns=["ein"])) == len(model_candidates)
print("Output validation passed.")


---
## 12 — Supplemental analysis: filing conflicts, repeat disclosures, and reporting patterns

These sections are **read-only** and can run while Notebook 03 processes other IRS archives. They load only the already-cleaned Parquet files created by Notebook 02; no ZIP archives are opened, and no existing modeling dataset is overwritten.

**To run independently:** execute the next setup cell, then Sections 13–15. Do **not** select “Run All” while Notebook 03 is busy. Dates and publication years are proxies for filing order, not verified amendment chronology. The target is a **self-reported significant diversion**, not verified fraud.

In [ ]:
# Standalone lightweight setup: run this cell even after restarting the kernel.
from pathlib import Path
import pandas as pd

here = Path.cwd().resolve()
ROOT = next((p for p in (here, *here.parents) if (p / ".gitignore").exists()), None)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project repository")
CLEANED = ROOT / "data" / "cleaned"
filing_path = CLEANED / "form990_filing_level_2023_2025.parquet"
canonical_path = CLEANED / "form990_organization_year_provisional_2023_2025.parquet"
for path in (filing_path, canonical_path):
    if not path.exists():
        raise FileNotFoundError(f"Run Notebook 02's original pipeline once to create: {path}")
filing_audit = pd.read_parquet(filing_path)
canonical_audit = pd.read_parquet(canonical_path)
print(f"Loaded {len(filing_audit):,} filings and {len(canonical_audit):,} organization-years (read-only).")

## 13 — Investigate conflicting organization-year disclosures

Compare the reported label in filing order, identify whether both labels appear within the same publication year, and flag cases where a positive disclosure is followed by a negative one or vice versa. **Publication year alone does not establish which return was actually filed first.** The provisional order below uses tax-period end, publication year, and filing ID for consistency with the existing canonical selection rule.

In [ ]:
keys = ["ein", "tax_year"]
conflict_keys = canonical_audit.loc[canonical_audit["has_conflicting_labels"].fillna(False), keys]
conflict_filings = filing_audit.merge(conflict_keys, on=keys, how="inner", validate="many_to_one").copy()
conflict_filings["_period_end"] = pd.to_datetime(conflict_filings["tax_period_end"], errors="coerce")
conflict_filings["_pub"] = pd.to_numeric(conflict_filings["publication_year"], errors="coerce")
conflict_filings["_id"] = conflict_filings["filing_id"].fillna("").astype(str)
conflict_filings = conflict_filings.sort_values(
    keys + ["_period_end", "_pub", "_id", "archive_name", "source_xml"],
    kind="stable", na_position="first"
)

def conflict_summary(g):
    labels = g["fraud_label"].dropna().astype(int).tolist()
    years = sorted(g["publication_year"].dropna().unique().tolist())
    first, last = (labels[0], labels[-1]) if labels else (None, None)
    return pd.Series({
        "organization_name": g["organization_name"].dropna().iloc[-1] if g["organization_name"].notna().any() else None,
        "filings": len(g), "publication_years": ", ".join(map(str, years)),
        "different_publication_years": len(years) > 1,
        "earliest_ordered_label": first, "latest_ordered_label": last,
        "ordered_transition": f"{first} → {last}",
        "label_sequence": " → ".join(map(str, labels)),
    })

if conflict_filings.empty:
    print("No conflicting groups in the current cleaned dataset.")
else:
    conflict_overview = conflict_filings.groupby(keys, sort=False).apply(
        conflict_summary
    ).reset_index()
    print(f"Conflicting groups: {len(conflict_overview):,}")
    print("Different publication years:", int(conflict_overview["different_publication_years"].sum()))
    display(conflict_overview)
    display(conflict_filings[["ein", "tax_year", "organization_name", "filing_id",
        "publication_year", "tax_period_end", "fraud_label", "raw_diversion_response", "archive_name"]])
print("Caution: these are ordering proxies, not verified original/amended return sequences.")

## 14 — Organizations with positive disclosures across multiple tax years

Count **distinct tax years per EIN**, not filings, so duplicate filings do not inflate repeat-disclosure counts. This uses the conservative canonical records and excludes the 16 ambiguous organization-year groups. Results therefore describe **repeat disclosures in the analyzed tax years**, not necessarily repeat diversion incidents.

In [ ]:
eligible = canonical_audit.loc[~canonical_audit["has_conflicting_labels"].fillna(False)].copy()
positive_years = eligible.loc[eligible["fraud_label"].eq(1),
    ["ein", "tax_year", "organization_name", "filing_id"]].drop_duplicates(["ein", "tax_year"])
repeat_orgs = (positive_years.groupby("ein", as_index=False)
    .agg(positive_tax_year_count=("tax_year", "nunique"),
         positive_tax_years=("tax_year", lambda s: ", ".join(map(str, sorted(s.unique())))),
         organization_name=("organization_name", "last"))
    .query("positive_tax_year_count > 1")
    .sort_values(["positive_tax_year_count", "ein"], ascending=[False, True]))
print(f"Positive organization-years (unambiguous): {len(positive_years):,}")
print(f"Distinct EINs with positive disclosures: {positive_years['ein'].nunique():,}")
print(f"EINs positive in more than one tax year: {len(repeat_orgs):,}")
display(repeat_orgs)
print("Tax years 2023–2025 have unequal publication coverage; 2025 is partial.")

## 15 — Disclosure rates by publication year and filing frequency

The first table is **filing-level** (multiple filings from one organization may appear). The second is **organization-year-level**, comparing groups with one filing against groups with multiple filings. These are descriptive associations, not causal estimates. A filing's publication year is not its tax year.

In [ ]:
def disclosure_table(frame, groups):
    out = (frame.groupby(groups, dropna=False)
        .agg(records=("fraud_label", "size"),
             positive=("fraud_label", lambda s: int(s.eq(1).sum())),
             negative=("fraud_label", lambda s: int(s.eq(0).sum())))
        .reset_index())
    out["positive_rate_pct"] = (100 * out["positive"] / out["records"]).round(4)
    return out

print("Filing-level disclosure rates by publication year and tax year:")
display(disclosure_table(filing_audit, ["publication_year", "tax_year"]).sort_values(["publication_year", "tax_year"]))

eligible["filing_frequency"] = eligible["filing_count"].apply(lambda x: "Multiple filings" if x > 1 else "Single filing")
print("Unambiguous organization-year disclosure rates by filing frequency:")
display(disclosure_table(eligible, ["tax_year", "filing_frequency"]).sort_values(["tax_year", "filing_frequency"]))

print("All-year organization-year summary:")
display(disclosure_table(eligible, ["filing_frequency"]))
print("Interpretation caution: repeated filings can reflect amendments, filing practices, or other causes.")

## 16 — Key Findings and Limitations

### Key Findings

- **Dataset size:** Processed 841,900 Form 990 filings, identifying 809 positive significant-diversion disclosures (0.0961%).
- **Modeling dataset:** After provisional deduplication and exclusion of conflicting disclosures, 829,027 organization-year records remain, including 776 positive disclosures.
- **Conflicting disclosures:** 16 organization-year groups contain both positive and negative responses. Eight have filings published in different years; the remaining eight have conflicting responses within the same publication year.
- **Repeated disclosures:** 113 organizations reported significant diversions in multiple tax years, including some with positive disclosures in all three years (2023–2025).
- **Publication timing:** Tax year 2023 filings published in 2026 had a higher positive disclosure rate (0.5106%) than earlier publication cohorts (approximately 0.09%).
- **Filing frequency:** Organization-years with multiple filings had a slightly higher positive disclosure rate (0.1142%) than those with a single filing (0.0933%).

### Limitations

- **Disclosure does not equal proven fraud:** A positive response indicates a self-reported significant diversion of assets, not independently verified fraud. A negative response does not establish that no fraud occurred.
- **Extreme class imbalance:** Positive disclosures represent approximately 0.1% of filings, requiring appropriate modeling and evaluation methods.
- **Incomplete tax-year coverage:** Tax year 2025 is only partially represented because additional filings may be published later.
- **Unverified filing chronology:** Canonical organization-year selection is provisional. Filing order does not establish amendment status or identify the authoritative return.
- **Repeated incidents:** Multiple positive disclosures from the same organization may reflect separate incidents, ongoing issues, or repeated reporting of an earlier incident.
- **No causal conclusions:** Differences in disclosure rates do not establish that filing frequency, publication timing, or organizational characteristics cause asset diversion.

## 17 — Modeling Considerations and Reproducibility

### Target Variable and Interpretation

The existing `fraud_label` variable represents whether an organization self-reported a significant diversion of assets under Form 990, Part VI, Section A, Line 5.

For interpretation, **diversion disclosure** is more accurate terminology than **fraud**, since the label does not independently establish criminal misconduct.

### Preventing Data Leakage

- Exclude `fraud_label` and disclosure-derived variables such as `ever_positive`, `positive_filing_count`, and `distinct_labels` from predictor columns.
- Do not use fields that directly reveal the organization's diversion disclosure or related explanatory information as predictive features.
- Retain EIN for entity identification and train/test splitting, but exclude it from the default predictive feature set.
- Use EIN-grouped train/test separation to prevent the same organization from appearing in both training and test datasets.
- Distinguish models predicting contemporaneous disclosures from models intended to predict future disclosures.

### Handling Conflicting and Repeated Filings

- Continue excluding the 16 conflicting organization-year groups from the conservative modeling dataset until filing histories can be investigated.
- Preserve all original filings and disclosure responses for auditing.
- Do not assume that the latest publication year represents the authoritative amended return.
- Investigate whether repeated positive disclosures represent distinct events or repeated reporting of the same incident.

### Evaluation Strategy

Because positive disclosures are extremely rare, overall accuracy will not be a sufficient performance metric.

Future modeling should emphasize:

- Precision and recall
- Precision-recall AUC
- False-positive rates and operational usefulness
- Comparisons against simple baseline models
- Performance across tax years and organizational subgroups

### Reproducibility

The supplemental analyses in Notebook 02 read existing Parquet datasets without modifying the saved filing-level, canonical organization-year, or modeling candidate datasets.

All transformations and exclusions should remain documented so that results can be reproduced from the original IRS XML archives.

### Next Steps

1. Investigate conflicting and repeated disclosures using additional filing information.
2. Extract financial, compensation, governance, and organizational features from the original XML.
3. Develop financial ratios and year-over-year change metrics.
4. Explore supervised classification, anomaly detection, and entity/network analysis.
5. Evaluate models using organization-level separation and appropriate methods for extreme class imbalance.